<a href="https://colab.research.google.com/github/nuurceng/CXR_26/blob/main/roi%26raw_swin_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!rm -rf /content/final_data

In [ ]:
import os, zipfile, torch, wandb, datetime, shutil
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
from transformers import SwinForImageClassification
from google.colab import drive, runtime

model_variant = "Swin-B" # Swin-T, Swin-S, Swin-B
data_type = "ROI"       # RAW, ROI
model_full_name = f"{model_variant}-{data_type}--{2}"

base_drive_path = "/content/drive/MyDrive/segmantation_chest_x-rays/lungsClassification-roi"
cm_drive_path = "/content/drive/MyDrive/segmantation_chest_x-rays/lungsClassificationCM-roi"
os.makedirs(base_drive_path, exist_ok=True)
os.makedirs(cm_drive_path, exist_ok=True)


In [ ]:
extract_path = '/content/final_data'
if not os.path.exists(extract_path):
    print("🚀 Veri seti yerel diske çıkarılıyor...")
    zip_path = '/content/drive/MyDrive/segmantation_chest_x-rays/FINAL_CLASSIFICATION_DATASET.zip'
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(extract_path)
print("🚀 Veri seti yerel diske çıkarılması tamamlandı...")

🚀 Veri seti yerel diske çıkarılıyor...
🚀 Veri seti yerel diske çıkarılması tamamlandı...


In [ ]:
train_dir = os.path.join(extract_path, 'train')
val_dir = os.path.join(extract_path, 'val')
test_dir = os.path.join(extract_path, 'test')

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
      'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

train_dataset = datasets.ImageFolder(train_dir, data_transforms['train'])
val_dataset = datasets.ImageFolder(val_dir, data_transforms['val'])
test_dataset = datasets.ImageFolder(test_dir, data_transforms['test'])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


In [ ]:
def plot_and_save_cm(y_true, y_pred, class_names, title, filename):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(10, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(xticks_rotation=45, ax=ax, values_format='d')#cmap='viridis',
    plt.title(title)
    plt.tight_layout()
    local_path = f"/content/{filename}"
    plt.savefig(local_path)
    plt.close()

    shutil.copyfile(local_path, os.path.join(cm_drive_path, filename))
    return local_path

wandb.login(key="key")
current_time = datetime.datetime.now().strftime("%H%M%S")
run_name = f"{model_full_name}-20Epoch-{current_time}"

run = wandb.init(
    project="Lung-Segmentation_Classification-roi",
    name=run_name,
    config={"model": model_variant, "data": data_type, "lr": 5e-5, "epochs": 20}
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SwinForImageClassification.from_pretrained(
    "microsoft/swin-base-patch4-window7-224",
    num_labels=4,
    ignore_mismatched_sizes=True
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.05)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=3)

# ---
best_f1 = 0.0
class_names = ['COVID', 'NORMAL', 'PNEUMONIA', 'TUBERCULOSIS']

for epoch in range(20):
    model.train()
    train_loss = 0
    all_train_preds, all_train_labels = [], []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/20 [Train]")
    for imgs, labels in pbar:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        all_train_preds.extend(preds.cpu().numpy())
        all_train_labels.extend(labels.cpu().numpy())
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_train_loss = train_loss / len(train_loader)
    t_acc = accuracy_score(all_train_labels, all_train_preds)
    t_prec, t_rec, t_f1, _ = precision_recall_fscore_support(all_train_labels, all_train_preds, average='macro')

    # ---
    model.eval()
    val_loss = 0
    all_val_preds, all_val_labels, all_val_probs = [], [], []

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs).logits
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)

            all_val_preds.extend(preds.cpu().numpy())
            all_val_labels.extend(labels.cpu().numpy())
            all_val_probs.extend(probs.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    v_acc = accuracy_score(all_val_labels, all_val_preds)
    v_prec, v_rec, v_f1, _ = precision_recall_fscore_support(all_val_labels, all_val_preds, average='macro')

    scheduler.step(v_f1)
    curr_lr = optimizer.param_groups[0]['lr']

    # ---
    wandb.log({
        "epoch": epoch + 1,
        "train/loss": avg_train_loss, "train/acc": t_acc, "train/f1": t_f1,
        "train/precision": t_prec, "train/recall": t_rec,
        "val/loss": avg_val_loss, "val/acc": v_acc, "val/f1": v_f1,
        "val/precision": v_prec, "val/recall": v_rec,
        "lr": curr_lr,
        "conf_mat_wandb": wandb.plot.confusion_matrix(probs=None, y_true=all_val_labels, preds=all_val_preds, class_names=class_names),
        "roc": wandb.plot.roc_curve(all_val_labels, np.array(all_val_probs), labels=class_names)
    })

    # ---
    print(f"\n" + "="*95)
    print(f"🚩 EPOCH {epoch+1} ÖZETİ (LR: {curr_lr:.2e})")
    print("-" * 95)
    print(f"TRAIN | Loss: {avg_train_loss:.4f} | Acc: {t_acc:.4f} | F1: {t_f1:.4f} | Prec: {t_prec:.4f} | Rec: {t_rec:.4f}")
    print(f"VAL   | Loss: {avg_val_loss:.4f} | Acc: {v_acc:.4f} | F1: {v_f1:.4f} | Prec: {v_prec:.4f} | Rec: {v_rec:.4f}")
    print("="*95 + "\n")

    # CHECKPOINT
    if v_f1 > best_f1:
        best_f1 = v_f1
        model_save_name = f"best_model_{model_full_name}.pth"
        local_model_path = f"/content/{model_save_name}"
        torch.save(model.state_dict(), local_model_path)
        shutil.copyfile(local_model_path, os.path.join(base_drive_path, model_save_name))
        print(f"⭐ En iyi model güncellendi! (F1: {best_f1:.4f} - Model: {model_save_name})")

# ---
print("\n🏆 EĞİTİM TAMAMLANDI. EN İYİ MODEL YÜKLENEREK FİNAL ANALİZ BAŞLIYOR...")
model.load_state_dict(torch.load(os.path.join(base_drive_path, f"best_model_{model_full_name}.pth")))
model.eval()

def final_report_and_visuals(loader, name):
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc=f"{name} Analiz Ediliyor"):
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs).logits
            probs = torch.softmax(outputs, dim=1); _, preds = torch.max(outputs, 1)
            all_labels.extend(labels.cpu().numpy()); all_preds.extend(preds.cpu().numpy()); all_probs.extend(probs.cpu().numpy())

    # A. Classification Report
    report = classification_report(all_labels, all_preds, target_names=class_names, digits=4)
    separator = "-" * 60
    final_text_report = f"\n📊 {name} SINIFLANDIRMA RAPORU\n{separator}\n{report}\n{separator}\n"
    print(final_text_report)

    run.summary[f"final_report_{name.lower().replace(' ', '_')}"] = final_text_report

    report_file_name = f"{model_full_name}_{name.replace(' ', '_')}_report.txt"
    report_file_path = os.path.join(base_drive_path, report_file_name)

    with open(report_file_path, "w", encoding="utf-8") as f:
        f.write(final_text_report)

    # B. Confusion Matrix
    cm_filename = f"CM_{model_full_name}_{name.replace(' ', '_')}.png"
    local_cm = plot_and_save_cm(all_labels, all_preds, class_names, f"{model_full_name} {name} CM", cm_filename)
    wandb.log({f"final_cm_{name.lower().replace(' ', '_')}": wandb.Image(local_cm)})

    # C. ROC Curve
    wandb.log({
        f"final_roc_{name.lower().replace(' ', '_')}": wandb.plot.roc_curve(
            all_labels,
            np.array(all_probs),
            labels=class_names,
            title=f"ROC Curve - {model_full_name} - {name}"
        )
    })


final_report_and_visuals(val_loader, "VALIDATION SETI")
final_report_and_visuals(test_loader, "TEST SETI")

artifact = wandb.Artifact(f"results-{model_full_name}", type="evaluation_results")

artifact.add_file(os.path.join(base_drive_path, f"best_model_{model_full_name}.pth"))

val_cm_name = f"CM_{model_full_name}_VALIDATION_SETI.png"
test_cm_name = f"CM_{model_full_name}_TEST_SETI.png"

if os.path.exists(os.path.join(cm_drive_path, val_cm_name)):
    artifact.add_file(os.path.join(cm_drive_path, val_cm_name))
if os.path.exists(os.path.join(cm_drive_path, test_cm_name)):
    artifact.add_file(os.path.join(cm_drive_path, test_cm_name))

val_rep_name = f"{model_full_name}_VALIDATION_SETI_report.txt"
test_rep_name = f"{model_full_name}_TEST_SETI_report.txt"

if os.path.exists(os.path.join(base_drive_path, val_rep_name)):
    artifact.add_file(os.path.join(base_drive_path, val_rep_name))
if os.path.exists(os.path.join(base_drive_path, test_rep_name)):
    artifact.add_file(os.path.join(base_drive_path, test_rep_name))

run.log_artifact(artifact)
print(f"📦 Artifact güncellendi: Model, CM'ler ve Raporlar WandB'ye paketlendi!")


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nurkaraca0638 (nurkaraca0638-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/352M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/449 [00:00<?, ?it/s]

...


🚩 EPOCH 20 ÖZETİ (LR: 5.00e-06)
-----------------------------------------------------------------------------------------------
TRAIN | Loss: 0.0001 | Acc: 1.0000 | F1: 1.0000 | Prec: 1.0000 | Rec: 1.0000
VAL   | Loss: 0.1584 | Acc: 0.9689 | F1: 0.9669 | Prec: 0.9663 | Rec: 0.9677

⭐ En iyi model güncellendi! (F1: 0.9669 - Model: best_model_Swin-B-ROI--2.pth)

🏆 EĞİTİM TAMAMLANDI. EN İYİ MODEL YÜKLENEREK FİNAL ANALİZ BAŞLIYOR...


VALIDATION SETI Analiz Ediliyor: 100%|██████████| 127/127 [00:40<00:00,  3.11it/s]



📊 VALIDATION SETI SINIFLANDIRMA RAPORU
------------------------------------------------------------
              precision    recall  f1-score   support

       COVID     0.9276    0.9613    0.9441       413
      NORMAL     0.9591    0.9418    0.9504       498
   PNEUMONIA     0.9840    0.9875    0.9858       561
TUBERCULOSIS     0.9945    0.9802    0.9873       556

    accuracy                         0.9689      2028
   macro avg     0.9663    0.9677    0.9669      2028
weighted avg     0.9693    0.9689    0.9690      2028

------------------------------------------------------------



TEST SETI Analiz Ediliyor: 100%|██████████| 64/64 [00:19<00:00,  3.24it/s]



📊 TEST SETI SINIFLANDIRMA RAPORU
------------------------------------------------------------
              precision    recall  f1-score   support

       COVID     0.9369    0.9324    0.9346       207
      NORMAL     0.9472    0.9320    0.9395       250
   PNEUMONIA     0.9755    0.9929    0.9841       281
TUBERCULOSIS     0.9857    0.9857    0.9857       279

    accuracy                         0.9636      1017
   macro avg     0.9613    0.9607    0.9610      1017
weighted avg     0.9635    0.9636    0.9635      1017

------------------------------------------------------------

📦 Artifact güncellendi: Model, CM'ler ve Raporlar WandB'ye paketlendi!


In [ ]:
wandb.finish()

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,███████████▁▁▁▁▁▁▁▁▁
train/acc,▁▄▆▆▇▇▇▇█▇▇█████████
train/f1,▁▄▆▆▇▇▇▇█▇▇█████████
train/loss,█▅▃▃▂▂▂▂▁▂▂▂▁▁▁▁▁▁▁▁
train/precision,▁▄▆▆▇▇▇▇█▇▇█████████
train/recall,▁▄▆▆▇▇▇▇█▇▇█████████
val/acc,▃▁▄▄▄▆▆▇▆▇▆▅████▇███
val/f1,▃▁▅▄▅▆▆▇▆▇▆▅████▇███
val/loss,▇█▅▅▆▂▅▃▅▁▅█▃▃▃▄▄▃▃▄
+2,...


In [ ]:
runtime.unassign()